In [3]:
import pandas as pd

# ==============================
# 1. ĐỌC DỮ LIỆU
# ==============================

orders = pd.read_csv(
    "ORDER.csv",
    encoding="utf-8-sig"
)

order_items = pd.read_csv(
    "ORDER_ITEMS.csv",
    encoding="utf-8-sig"
)


# ==============================
# 2. TÍNH GIÁ TRỊ TỪNG DÒNG HÀNG
# ==============================

order_items["line_revenue"] = (
    order_items["quantity"] * order_items["unit_price"]
    - order_items["discount_amount"]
)


# ==============================
# 3. TỔNG GIÁ TRỊ TỪNG HÓA ĐƠN
# ==============================

invoice_value = (
    order_items
    .groupby("order_id", as_index=False)["line_revenue"]
    .sum()
    .rename(columns={"line_revenue": "invoice_value"})
)


# ==============================
# 4. GHÉP ĐƠN HÀNG VỚI GIÁ TRỊ HÓA ĐƠN
# ==============================

data = orders.merge(
    invoice_value,
    on="order_id",
    how="left"
)

data["invoice_value"] = data["invoice_value"].fillna(0)


# ==============================
# 5. XÁC ĐỊNH HÓA ĐƠN HOÀN THÀNH
# ==============================

# Đơn hàng delivered được xem là giao dịch
# hoàn thành thành công.

data["successful_invoice"] = (
    data["order_status"] == "delivered"
).astype(int)


# ==============================
# 6. THỐNG KÊ THEO NHÂN VIÊN
# ==============================

employee_analysis = (
    data
    .groupby("sales_employee_id")
    .agg(
        total_invoices=("order_id", "nunique"),
        successful_invoices=("successful_invoice", "sum"),
        total_sales=("invoice_value", "sum"),
        average_sales_per_invoice=("invoice_value", "mean"),
        returned_invoices=(
            "order_status",
            lambda x: (x == "returned").sum()
        ),
        cancelled_invoices=(
            "order_status",
            lambda x: (x == "cancelled").sum()
        )
    )
    .reset_index()
)


# ==============================
# 7. TÍNH TỶ LỆ CHUYỂN ĐỔI
# ==============================

employee_analysis["conversion_rate"] = (
    employee_analysis["successful_invoices"]
    / employee_analysis["total_invoices"]
    * 100
)


# ==============================
# 8. TÍNH CHẤT LƯỢNG GIAO DỊCH
# ==============================

employee_analysis["transaction_quality"] = (
    employee_analysis["successful_invoices"]
    / employee_analysis["total_invoices"]
    * 100
)


# ==============================
# 9. XẾP HẠNG NHÂN VIÊN
# ==============================

employee_analysis["rank_conversion"] = (
    employee_analysis["conversion_rate"]
    .rank(method="min", ascending=False)
    .astype(int)
)

employee_analysis["rank_average_invoice"] = (
    employee_analysis["average_sales_per_invoice"]
    .rank(method="min", ascending=False)
    .astype(int)
)

employee_analysis["rank_transaction_quality"] = (
    employee_analysis["transaction_quality"]
    .rank(method="min", ascending=False)
    .astype(int)
)


# ==============================
# 10. SẮP XẾP KẾT QUẢ
# ==============================

employee_analysis = employee_analysis.sort_values(
    by=[
        "conversion_rate",
        "average_sales_per_invoice",
        "transaction_quality"
    ],
    ascending=False
)


# ==============================
# 11. HIỂN THỊ KẾT QUẢ
# ==============================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\n===== ĐÁNH GIÁ HIỆU QUẢ NHÂN VIÊN =====\n")

print(
    employee_analysis.to_string(index=False)
)


# ==============================
# 12. LƯU KẾT QUẢ
# ==============================

employee_analysis.to_csv(
    "problem4_statement5_employee_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nĐã lưu kết quả vào: "
    "problem4_statement5_employee_analysis.csv"
)



===== ĐÁNH GIÁ HIỆU QUẢ NHÂN VIÊN =====

sales_employee_id  total_invoices  successful_invoices  total_sales  average_sales_per_invoice  returned_invoices  cancelled_invoices  conversion_rate  transaction_quality  rank_conversion  rank_average_invoice  rank_transaction_quality
          EMP0133            3281                 2679  81587125.82               24866.542463                179                 260        81.651935            81.651935                1                    48                         1
          EMP0124            3244                 2645  77607965.72               23923.540604                165                 283        81.535142            81.535142                2                   193                         2
          EMP0163            3218                 2623  82061939.93               25500.913589                172                 272        81.510255            81.510255                3                     2                         3
          